In [ ]:
from jupyter_dash import JupyterDash

import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

import os

import numpy as np
import pandas as pd

from CRUD_Python_Module import AnimalShelter

db = AnimalShelter()

db.create_all_indexes()

LAT_COL = 'location_lat'
LONG_COL = 'location_long'
BREED_COL = 'breed'
NAME_COL = 'name'

RESCUE_BREEDS = [
    'Labrador Retriever Mix',
    'Chesapeake Bay Retriever',
    'Newfoundland',
]

RESCUE_QUERIES = {
    'Water Rescue': {
        'breed': {'$in': RESCUE_BREEDS},
        'sex_upon_outcome': 'Intact Female',
        'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
    },
    'Mountain or Wilderness Rescue': {
        'breed': {'$in': RESCUE_BREEDS},
        'sex_upon_outcome': 'Intact Male',
        'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
    },
    'Disaster or Individual Tracking': {
        'breed': {'$in': RESCUE_BREEDS},
        'sex_upon_outcome': 'Intact Male',
        'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300},
    },
}

def records_to_frame(records):
    frame = pd.DataFrame.from_records(records)
    return frame.drop(columns=['_id'], errors='ignore')

df = records_to_frame(db.read({}))

REPORT_ROW_LIMIT = 15
REPORT_MIN_COUNT = 5

def load_breed_outcome_report():
    rows = db.get_breed_outcome_report(min_count=REPORT_MIN_COUNT,
                                       limit=REPORT_ROW_LIMIT)
    if not rows:
        return pd.DataFrame(columns=['breed', 'outcome_type', 'total'])
    return pd.DataFrame.from_records(rows)

app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display': 'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
    html.A(
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode())),
        href='https://www.snhu.edu',
        target='_blank'
    ),
    html.Div([
        html.Label("Filter by Animal Type:"),
        dcc.Dropdown(
            id='filter-type',
            options=(
                [{'label': 'All', 'value': 'all'}]
                + [{'label': name, 'value': name} for name in RESCUE_QUERIES]
            ),
            value='all',
            multi=False
        )
    ]),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True}
                 for i in df.columns],
        data=df.to_dict('records'),
        row_selectable="single",
        selected_rows=[0],
        page_size=15,
        sort_action="native",
        filter_action="native",
        page_action="native"
    ),
    html.Br(),
    html.Hr(),
    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(
                     id='graph-id',
                     className='col s12 m6',
                 ),
                 html.Div(
                     id='map-id',
                     className='col s12 m6',
                 )
             ]),
    html.Footer("Dustin Ledbetter",
                style={'text-align': 'left'}),
    html.Hr(),
    html.Center(html.H2('Breed and Outcome Report')),

    html.Div(id='report-summary-id', style={'text-align': 'center'}),
    html.Div(id='report-chart-id'),
    dcc.Interval(id='report-refresh-id', interval=300000, n_intervals=0),
])

@app.callback(Output('datatable-id', 'data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    query = RESCUE_QUERIES.get(filter_type, {})
    filtered = records_to_frame(db.read(query))
    return filtered.to_dict('records')

@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if not viewData:
        return [html.P('No records match the current filter.')]

    dff = pd.DataFrame.from_dict(viewData)
    if BREED_COL not in dff.columns or dff.empty:
        return [html.P('No breed data available for the current filter.')]

    return [
        dcc.Graph(
            figure=px.pie(dff, names=BREED_COL, title='Preferred Animals')
        )
    ]

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns or []]

@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if not viewData:
        return [html.P('No records to map.')]

    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return [html.P('No records to map.')]

    row = index[0] if index else 0
    if row >= len(dff):
        return [html.P('No records to map.')]

    required = {LAT_COL, LONG_COL}
    if not required.issubset(dff.columns):
        return [html.P('Location data is unavailable for these records.')]

    latitude = dff.iloc[row][LAT_COL]
    longitude = dff.iloc[row][LONG_COL]
    if pd.isna(latitude) or pd.isna(longitude):
        return [html.P('This record has no coordinates.')]

    breed = dff.iloc[row].get(BREED_COL, 'Unknown breed')
    name = dff.iloc[row].get(NAME_COL, 'Unknown')

    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[latitude, longitude], children=[
                dl.Tooltip(breed),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(name)
                ])
            ])
        ])
    ]

@app.callback(
    Output('report-chart-id', 'children'),
    [Input('report-refresh-id', 'n_intervals')])
def update_report_chart(_n_intervals):
    report = load_breed_outcome_report()
    if report.empty:
        return [html.P('No outcome data is available to report on.')]

    return [
        dcc.Graph(
            figure=px.bar(
                report,
                x='breed',
                y='total',
                color='outcome_type',
                barmode='group',
                title='Top breed and outcome combinations (min %d records)'
                      % REPORT_MIN_COUNT,
                labels={'total': 'Animals',
                        'breed': 'Breed',
                        'outcome_type': 'Outcome'},
            )
        )
    ]

@app.callback(
    Output('report-summary-id', 'children'),
    [Input('report-refresh-id', 'n_intervals')])
def update_report_summary(_n_intervals):
    counts = db.get_rescue_candidate_counts(RESCUE_QUERIES)
    if not counts:
        return html.P('Candidate counts are unavailable.')

    return html.P(' | '.join(
        '%s: %d candidates' % (name, total) for name, total in counts.items()
    ))

app.run_server()
